In [ ]:
!pip install datasets pandas rapidfuzz

In [ ]:
import csv
import os
import re
import warnings

import numpy as np
import pandas as pd
from datasets import load_dataset
from dotenv import find_dotenv, load_dotenv
from huggingface_hub import login
from rapidfuzz import fuzz, process
from scipy.spatial import distance
from scipy.stats import ks_2samp, mannwhitneyu
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
root_path = "/content/drive/MyDrive/BTL_AI_13_IT2302/AIWritingIndicator-13/SourceCode/"
%cd {root_path}
!pwd

/content/drive/.shortcut-targets-by-id/17JBaRqvfDFY0j8cN0vwu8kzqwtDISdDR/BTL_AI_13_IT2302/AIWritingIndicator-13/SourceCode
/content/drive/.shortcut-targets-by-id/17JBaRqvfDFY0j8cN0vwu8kzqwtDISdDR/BTL_AI_13_IT2302/AIWritingIndicator-13/SourceCode


In [ ]:
env_path = find_dotenv()

if env_path:
    load_dotenv(env_path)
else:
    print("Không tìm thấy file .env!")

In [ ]:
hf_token = os.getenv("HF_TOKEN")

if hf_token:
    login()
else:
    print("Không tồn tại env HF_TOKEN!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
print("Đang tải dataset...")
dataset = load_dataset("ICCIES-2025-DetectAI/vietnamese_news_human_ai", split="train")
df = dataset.to_pandas()

df_label_0 = df[df['Label'] == 0].copy()
df_label_1 = df[df['Label'] == 1].copy()

def process_and_filter(text):
    if not isinstance(text, str):
        return None

    text = text.strip()
    if not text:
        return None

    if not text[0].isalpha():
        return None
    if not text[0].isupper():
        text = text[0].upper() + text[1:]

    if ":" in text:
        return None

    text = re.sub(r'[\n\r\t]+', ' ', text)

    words = text.split()
    if len(words) < 200 or len(words) > 2000:
        return None

    if re.search(r'(http[s]?://|www\.|<.*?>|\S+@\S+|#\w+|@\w+)', text):
        return None

    if "  " in text:
        return None

    allowed_pattern = re.compile(r'^[a-zA-Z0-9\s\.,\?!\-\(\)\'"“”‘’/_%a-zA-ZàáãạảăắằẳẵặâấầẩẫậèéẹẻẽêềếểễệđìíĩỉịòóõọỏôốồổỗộơớờởỡợùúũụủưứừửữựỳýỹỷỵÀÁÃẠẢĂẮẰẲẴẶÂẤẦẨẪẬÈÉẸẺẼÊỀẾỂỄỆĐÌÍĨỈỊÒÓÕỌỎÔỐỒỔỖỘƠỚỜỞỠỢÙÚŨỤỦƯỨỪỬỮỰỲÝỸỶỴ]+$')
    if not allowed_pattern.match(text):
        return None

    upper_chars = sum(1 for c in text if c.isupper())
    if (upper_chars / len(text)) > 0.2:
        return None

    vietnamese_vowels = re.findall(r'[àáãạảăắằẳẵặâấầẩẫậèéẹẻẽêềếểễệđìíĩỉịòóõọỏôốồổỗộơớờởỡợùúũụủưứừửữựỳýỹỷỵ]', text.lower())
    if (len(vietnamese_vowels) / len(words)) < 0.2:
        return None

    if not text.endswith('.'):
        last_period_index = text.rfind('.')
        if last_period_index == -1:
            return None
        text = text[:last_period_index + 1]

    if text.count('"') % 2 != 0:
        return None

    if len(text.split()) < 200:
        return None

    return text

print("Đang lọc nhiễu Label 0 (Human)...")
df_label_0['processed_text'] = df_label_0['Text'].apply(process_and_filter)
df_clean_0 = df_label_0.dropna(subset=['processed_text']).copy()
df_clean_0['Text'] = df_clean_0['processed_text']
df_clean_0 = df_clean_0.drop(columns=['processed_text'])

print("Đang lọc nhiễu Label 1 (AI)...")
df_label_1['processed_text'] = df_label_1['Text'].apply(process_and_filter)
df_clean_1 = df_label_1.dropna(subset=['processed_text']).copy()
df_clean_1['Text'] = df_clean_1['processed_text']
df_clean_1 = df_clean_1.drop(columns=['processed_text'])

print(f"Số mẫu CỰC SẠCH thu được: Label 0 = {len(df_clean_0)} | Label 1 = {len(df_clean_1)}")
print("Đang đồng bộ hóa phân bố độ dài văn bản...")
df_clean_0['word_count'] = df_clean_0['Text'].apply(lambda x: len(x.split()))
df_clean_1['word_count'] = df_clean_1['Text'].apply(lambda x: len(x.split()))
bins = np.arange(200, 2051, 50)
labels = [f"{bins[i]}-{bins[i+1]-1}" for i in range(len(bins)-1)]
df_clean_0['length_bin'] = pd.cut(df_clean_0['word_count'], bins=bins, labels=labels, right=False)
df_clean_1['length_bin'] = pd.cut(df_clean_1['word_count'], bins=bins, labels=labels, right=False)
target_counts = df_clean_0['length_bin'].value_counts()
sampled_df1_list = []
for bin_label, target_n in target_counts.items():
    if target_n == 0:
        continue

    pool_1 = df_clean_1[df_clean_1['length_bin'] == bin_label]
    available_n = len(pool_1)

    if available_n >= target_n:
        sampled_df1_list.append(pool_1.sample(n=target_n, random_state=42))
    else:
        if available_n > 0:
            print(f"  -> [Cảnh báo] Giỏ {bin_label} từ: Cần {target_n} mẫu nhưng Label 1 chỉ có {available_n}. Lấy toàn bộ.")
            sampled_df1_list.append(pool_1)

df_balanced_1 = pd.concat(sampled_df1_list)

sampled_df0_list = []
for bin_label, count_1 in df_balanced_1['length_bin'].value_counts().items():
    if count_1 > 0:
        pool_0 = df_clean_0[df_clean_0['length_bin'] == bin_label]
        sampled_df0_list.append(pool_0.sample(n=count_1, random_state=42))

df_balanced_0 = pd.concat(sampled_df0_list)
df_final = pd.concat([df_balanced_0, df_balanced_1]).sample(frac=1, random_state=42).reset_index(drop=True)
df_final = df_final.drop(columns=['word_count', 'length_bin'])

print(f"\n HOÀN TẤT! Số mẫu cuối cùng: Label 0 = {len(df_balanced_0)} | Label 1 = {len(df_balanced_1)}")
print("Phân bố độ dài đã được ép khớp 100%.")

df_final.to_csv("Data/iccies_dataset_testing/balanced_super_strict_dataset.csv",
                index=False,
                encoding='utf-8-sig',
                quoting=csv.QUOTE_ALL)
print("Đã lưu tập dữ liệu cân bằng tuyệt đối ra file: balanced_super_strict_dataset.csv")

Đang tải dataset...
Đang lọc nhiễu Label 0 (Human)...
Đang lọc nhiễu Label 1 (AI)...
Số mẫu CỰC SẠCH thu được: Label 0 = 11706 | Label 1 = 29436
Đang đồng bộ hóa phân bố độ dài văn bản...
  -> [Cảnh báo] Giỏ 200-249 từ: Cần 5408 mẫu nhưng Label 1 chỉ có 41. Lấy toàn bộ.
  -> [Cảnh báo] Giỏ 250-299 từ: Cần 3271 mẫu nhưng Label 1 chỉ có 3005. Lấy toàn bộ.
  -> [Cảnh báo] Giỏ 450-499 từ: Cần 78 mẫu nhưng Label 1 chỉ có 73. Lấy toàn bộ.
  -> [Cảnh báo] Giỏ 500-549 từ: Cần 22 mẫu nhưng Label 1 chỉ có 19. Lấy toàn bộ.

 HOÀN TẤT! Số mẫu cuối cùng: Label 0 = 6065 | Label 1 = 6065
Phân bố độ dài đã được ép khớp 100%.
Đã lưu tập dữ liệu cân bằng tuyệt đối ra file: balanced_super_strict_dataset.csv


In [ ]:
class DatasetMergeEvaluator:
    def __init__(self, path_a, path_b):
        print("Đang tải dữ liệu...")
        self.df_a = pd.read_csv(path_a)
        self.df_b = pd.read_csv(path_b)

        rename_map = {}
        if 'Text' in self.df_a.columns: rename_map['Text'] = 'text'
        if 'Label' in self.df_a.columns: rename_map['Label'] = 'label'

        if rename_map:
            self.df_a = self.df_a.rename(columns=rename_map)
            print(f"Đã chuẩn hóa tên cột cho Dataset A: {list(rename_map.keys())} -> {list(rename_map.values())}")

        self.df_a = self.df_a.dropna(subset=['text'])
        self.df_b = self.df_b.dropna(subset=['text'])

        print(f"Dataset A: {len(self.df_a)} mẫu.")
        print(f"Dataset B: {len(self.df_b)} mẫu.")

        self.embed_model = None

    def _get_text_stats(self, text):
        words = str(text).split()
        sentences = re.split(r'[.!?]+', str(text))
        paragraphs = str(text).split('\n')
        return len(words), len(sentences), len(paragraphs)

    def check_1_basic_statistics(self):
        print("\n1. KIỂM TRA PHÂN PHỐI THỐNG KÊ CƠ BẢN")
        stats_a = self.df_a['text'].apply(self._get_text_stats).tolist()
        stats_b = self.df_b['text'].apply(self._get_text_stats).tolist()

        len_a = [x[0] for x in stats_a]
        len_b = [x[0] for x in stats_b]

        print(f"  - Độ dài từ (Dataset A): Mean={np.mean(len_a):.2f}, Var={np.var(len_a):.2f}")
        print(f"  - Độ dài từ (Dataset B): Mean={np.mean(len_b):.2f}, Var={np.var(len_b):.2f}")

        stat, p_val = ks_2samp(len_a, len_b)
        print(f"  -> KS-Test trên độ dài từ: p-value = {p_val:.4f}")
        if p_val > 0.05:
            print("  => Phân phối độ dài bài viết TƯƠNG ĐỒNG (Không khác biệt ý nghĩa thống kê).")
        else:
            print("  => Phân phối độ dài bài viết KHÁC BIỆT.")

    def check_2_dataset_separability(self):
        print("\n2. KIỂM TRA DATASET SEPARABILITY")
        a_texts = self.df_a['text'].tolist()
        b_texts = self.df_b['text'].tolist()

        X = a_texts + b_texts
        y = [0]*len(a_texts) + [1]*len(b_texts)

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        vectorizer = TfidfVectorizer(max_features=5000)
        X_train_vec = vectorizer.fit_transform(X_train)
        X_test_vec = vectorizer.transform(X_test)

        clf = LogisticRegression(max_iter=1000)
        clf.fit(X_train_vec, y_train)

        preds = clf.predict(X_test_vec)
        acc = accuracy_score(y_test, preds)
        print(f"  - Classifier Accuracy phân biệt A và B: {acc:.2%}")

        if acc < 0.60:
            print("  => Rất tốt để hợp nhất (Accuracy ~ 50%).")
        elif acc <= 0.75:
            print("  => Có thể hợp nhất (Cần cân nhắc thêm trọng số/bias).")
        else:
            print("  => CẢNH BÁO: Hai dataset rất khác nhau. Hợp nhất dễ gây bias mạnh!")

    def _get_embeddings(self, texts, sample_size=1000):
        if self.embed_model is None:
            print("  [Đang tải SentenceTransformer 'keepitreal/vietnamese-sbert' ...]")
            self.embed_model = SentenceTransformer('keepitreal/vietnamese-sbert')

        sample_texts = pd.Series(texts).sample(min(len(texts), sample_size), random_state=42).tolist()
        return self.embed_model.encode(sample_texts, show_progress_bar=False)

    def check_3_embedding_distance(self):
        print("\n3. ĐO KHOẢNG CÁCH PHÂN PHỐI BẰNG EMBEDDING")
        emb_a = self._get_embeddings(self.df_a['text'].tolist())
        emb_b = self._get_embeddings(self.df_b['text'].tolist())

        centroid_a = np.mean(emb_a, axis=0)
        centroid_b = np.mean(emb_b, axis=0)

        cos_sim = cosine_similarity([centroid_a], [centroid_b])[0][0]
        js_dist = distance.jensenshannon(np.abs(centroid_a), np.abs(centroid_b))

        print(f"  - Cosine Similarity giữa Centroids: {cos_sim:.4f} (Càng gần 1 càng tốt)")
        print(f"  - Jensen-Shannon Distance: {js_dist:.4f} (Càng gần 0 càng tốt)")

    def check_4_duplication(self, sample_size=500):
        print("\n4. KIỂM TRA TRÙNG LẶP (Duplication & Leakage)")
        b_samples = self.df_b['text'].sample(min(len(self.df_b), sample_size), random_state=42).tolist()
        a_texts = self.df_a['text'].tolist()

        exact_matches = 0
        fuzzy_matches = 0

        for text in b_samples:
            if text in a_texts:
                exact_matches += 1
            else:
                match = process.extractOne(text, a_texts, scorer=fuzz.ratio)
                if match and match[1] > 90:
                    fuzzy_matches += 1

        print(f"  - Exact Match (Trùng lặp hoàn toàn): {exact_matches}/{sample_size}")
        print(f"  - Near-Duplication (Trùng lặp > 90%): {fuzzy_matches}/{sample_size}")

    def evaluate_all(self):
        self.check_1_basic_statistics()
        self.check_2_dataset_separability()
        self.check_3_embedding_distance()
        self.check_4_duplication()

        print("\nKẾT LUẬN TỔNG QUAN:")
        print("  - Nếu Classifier Accuracy > 80%: Hai dataset có phong cách/đặc trưng quá khác biệt.")
        print("  - Nếu cột 'origin' ở Dataset B không cần thiết cho việc huấn luyện, bạn có thể gộp như sau:")
        print("    df_merged = pd.concat([df_a[['text', 'label']], df_b[['text', 'label']]], ignore_index=True)")

In [ ]:
path_A = "Data/iccies_dataset_testing/balanced_super_strict_dataset.csv"
path_B = "Data/vietnamese_news_ai_dataset_filtered.csv"
evaluator = DatasetMergeEvaluator(path_A, path_B)
evaluator.evaluate_all()